# UCI Appliances Energy Prediction — processing

Source: UCI ML Repository / Candanedo et al. (2017), `energydata_complete.csv`. Single house, 10-minute sampling, 19,735 rows, 29 columns (indoor temp/humidity per room + outdoor weather + target).

Two issues found on inspection:
- `rv1` and `rv2` are identical in every row (fraction equal = 1.0) — these are the original authors' synthetic random-noise control columns, not real measurements. Dropped.
- Sampling is 10 minutes, not hourly. Resampled to 1h to match the study's window-length protocol (48h-168h -> 1h).

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
from common import report_candidate, check_cumulative

RAW_PATH = "../data/raw/energydata_complete.csv"
PROCESSED_PATH = "../data/processed/uci_appliances_energy.csv"

df = pd.read_csv(RAW_PATH, parse_dates=["date"])
df.columns = [c.strip() for c in df.columns]
df = df.set_index("date")
df.index.name = "timestamp"

diffs = df.index.to_series().diff().dropna()
print(diffs.value_counts().head())
print("missing values:", df.isna().sum().sum())

In [ ]:
TARGET = "Appliances"

print("rv1 == rv2 fraction:", (df["rv1"] == df["rv2"]).mean())
df = df.drop(columns=["rv1", "rv2"])
feature_cols = [c for c in df.columns if c != TARGET]
print("feature count:", len(feature_cols))

In [ ]:
check_cumulative(df, [TARGET] + feature_cols).tail()

## Resample 10-min -> hourly

`Appliances`/`lights` are interval energy readings (Wh consumed in that 10-min slot, not cumulative) -> sum over the hour. Everything else is an instantaneous sensor reading (temperature, humidity, pressure, wind, etc.) -> mean over the hour. Partial hour bins (not exactly 6 sub-readings) are dropped rather than resampled from incomplete data.

In [ ]:
SUM_COLS = ["Appliances", "lights"]
MEAN_COLS = [c for c in feature_cols if c not in SUM_COLS]

hourly_sum = df[SUM_COLS].resample("1h").sum()
hourly_mean = df[MEAN_COLS].resample("1h").mean()
hourly = pd.concat([hourly_sum, hourly_mean], axis=1)[[TARGET] + feature_cols]

counts = df[SUM_COLS[0]].resample("1h").count()
print("rows per hour bin (should all be 6):")
print(counts.value_counts())

full_hours = counts[counts == 6].index
hourly = hourly.loc[full_hours].dropna(how="any")
hourly.shape

In [ ]:
report_candidate(hourly, TARGET, feature_cols, freq="1h", name="UCI Appliances Energy (hourly, processed)")

In [ ]:
hourly.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={hourly.shape}")